# 🤖 Notebook 04 — Model Training
## Bagian 4: Training Model ML + Experiment Runner

**Subset Prioritas:**
- Feature Extractors: TF-IDF, GloVe, FastText, Word2Vec, IndoBERT
- Models: Decision Tree, Random Forest, XGBoost, LightGBM, Logistic Regression, SVM

**Enriched Pooling:** mean+max+min+std untuk word embeddings

**Total kombinasi:** 30 kombinasi

## Setup

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import scipy.sparse as sp

from src.experiment_runner import run_experiments

## Automated Experiment Runner

Jalankan kombinasi model ringan dengan Word2Vec.

In [2]:
# Load dataset yang sudah dilabeli dan dipreprocessing
df = pd.read_csv('../data/processed/reviews_prepared_new.csv')
print(f"📊 Dataset: {df.shape[0]} baris")

# Pastikan kolom yang diperlukan ada
assert 'review_clean' in df.columns, "❌ Jalankan Notebook 02 terlebih dahulu!"
assert 'sentiment_encoded' in df.columns, "❌ Jalankan Notebook 02 terlebih dahulu!"

📊 Dataset: 1646 baris


In [3]:
# Jalankan experiment runner (Menggunakan dataset balanced 2024-2026)
print("\n🚀 Menjalankan Experiment Runner (Balanced Dataset 2024-2026)...")
print("   Extractors: TF-IDF, Word2Vec, FastText, Glove")
print("   Models: Logistic Regression, SVM, XGBoost, Random Forest, Decision Tree")
print("   ⏱️  Estimasi waktu: 10-20 menit\n")

# Gunakan data yang sudah diseimbangkan dan di-filter tahun 2024-2026
df_balanced = pd.read_csv('../data/processed/reviews_prepared_new.csv')

# Pastikan label berurutan [0, 1] jika Binary
if df_balanced['sentiment_encoded'].nunique() == 2:
    # Mapping: 0 tetap 0 (Negatif), 2 menjadi 1 (Positif)
    df_balanced['sentiment_encoded'] = df_balanced['sentiment_encoded'].map({0: 0, 2: 1})
    print("✅ Labels re-encoded to [0, 1] for Binary Classification")

comparison_df = run_experiments(
    df_balanced,
    text_col='review_clean', 
    label_col='sentiment_encoded',
    subset='priority',
    test_size=0.2,
    random_state=42,
    n_iter=10,
    cv=5,                   
    save_features=True,
    save_dir='../results'
)


🚀 Menjalankan Experiment Runner (Balanced Dataset 2024-2026)...
   Extractors: TF-IDF, Word2Vec, FastText, Glove
   Models: Logistic Regression, SVM, XGBoost, Random Forest, Decision Tree
   ⏱️  Estimasi waktu: 10-20 menit

✅ Labels re-encoded to [0, 1] for Binary Classification

🚀 EXPERIMENT RUNNER — Klasifikasi Sentimen CoreTax
   📌 Mode: BINARY Classification (Negatif vs Positif)

📊 Dataset split:
   Train: 1316 | Test: 330
   Train distribution: [658 658]
   Test distribution:  [165 165]

🔧 Extractors: ['TF-IDF', 'Word2Vec', 'FastText', 'GloVe']
🤖 Models: ['Logistic Regression', 'Random Forest', 'XGBoost', 'Decision Tree']

📈 Total kombinasi valid: 16
----------------------------------------------------------------------

📦 Feature Extractor: TF-IDF
   TF-IDF: vocab=1234, shape=(1316, 1234)
   💾 Features saved: tf-idf
   ⏱️  Extraction time: 0.0s

   [1/16] 🤖 TF-IDF + Logistic Regression
      ✅ W-F1=0.9060 | M-F1=0.9060 | Time=2.4s

   [2/16] 🤖 TF-IDF + Random Forest
      ✅ W-F1

In [4]:
# Tampilkan hasil
if comparison_df is not None:
    print("\n📊 Tabel Komparasi Lengkap:")
    print(comparison_df[['feature_extractor', 'model', 'weighted_f1', 'macro_f1',
                          'accuracy', 'train_time_s']].to_string())


📊 Tabel Komparasi Lengkap:
     feature_extractor                model  weighted_f1  macro_f1  accuracy  train_time_s
Rank                                                                                      
1               TF-IDF  Logistic Regression     0.906039  0.906039  0.906061          2.38
2             Word2Vec              XGBoost     0.903027  0.903027  0.903030        114.85
3               TF-IDF              XGBoost     0.902941  0.902941  0.903030          4.91
4             Word2Vec        Random Forest     0.902902  0.902902  0.903030         14.02
5               TF-IDF        Random Forest     0.893931  0.893931  0.893939          5.24
6                GloVe              XGBoost     0.890893  0.890893  0.890909        407.25
7             FastText        Random Forest     0.890873  0.890873  0.890909         15.37
8             Word2Vec  Logistic Regression     0.881808  0.881808  0.881818          9.04
9             FastText              XGBoost     0.872652  0.87

## Ringkasan Model Training

In [5]:
if comparison_df is not None and len(comparison_df) > 0:
    print("\n🏆 Top 3 Kombinasi Terbaik:")
    print("=" * 80)
    top3 = comparison_df.head(3)
    for idx, row in top3.iterrows():
        print(f"   #{idx+1}: {row['feature_extractor']} + {row['model']}")
        print(f"         W-F1={row['weighted_f1']:.4f} | M-F1={row['macro_f1']:.4f} | "
              f"Acc={row['accuracy']:.4f} | Train={row['train_time_s']:.1f}s")

    best = comparison_df.iloc[0]
    print(f"\n🥇 BEST: {best['feature_extractor']} + {best['model']}")
    print(f"   Weighted F1 = {best['weighted_f1']:.4f}")
    print(f"   Accuracy = {best['accuracy']:.4f}")
    
    # Analisis current performance
    avg_acc = comparison_df['accuracy'].mean()
    max_acc = comparison_df['accuracy'].max()
    print(f"\n📊 Current Performance:")
    print(f"   Average Accuracy: {avg_acc:.4f} ({avg_acc*100:.2f}%)")
    print(f"   Best Accuracy: {max_acc:.4f} ({max_acc*100:.2f}%)")
    print(f"   Target: 75% accuracy")


🏆 Top 3 Kombinasi Terbaik:
   #2: TF-IDF + Logistic Regression
         W-F1=0.9060 | M-F1=0.9060 | Acc=0.9061 | Train=2.4s
   #3: Word2Vec + XGBoost
         W-F1=0.9030 | M-F1=0.9030 | Acc=0.9030 | Train=114.8s
   #4: TF-IDF + XGBoost
         W-F1=0.9029 | M-F1=0.9029 | Acc=0.9030 | Train=4.9s

🥇 BEST: TF-IDF + Logistic Regression
   Weighted F1 = 0.9060
   Accuracy = 0.9061

📊 Current Performance:
   Average Accuracy: 0.8769 (87.69%)
   Best Accuracy: 0.9061 (90.61%)
   Target: 75% accuracy


In [6]:
print("\n✅ Model training selesai! Lanjut ke Notebook 05 untuk analisis komparatif.")


✅ Model training selesai! Lanjut ke Notebook 05 untuk analisis komparatif.
